# Infra-Bench v2 — Prithvi-EO-2.0-300M-TL Linear Probe (spatial split, 3 seeds)

Fifth and final FM in the Infra-Bench v2 cross-FM comparison, parallel
to the CROMA v2 / SatlasS2 v2 / SatlasS1 v2 / AlphaEarth v2 notebooks.
**Does not modify the v1 Prithvi notebook (`infra_fm_prithvi_eval.ipynb`).**

## Integration risk acknowledgment

Prithvi-EO-2.0 has historically been the most fragile integration in this
benchmark. Documented v1 failures:

1. **TerraTorch import fails in Colab** — scipy/dask/rapids dependency
   conflict produces `TypeError: xp_capabilities() got an unexpected
   keyword argument 'out_of_scope'`. The v1 notebook disabled the
   TerraTorch path entirely.
2. **`AutoModel.from_pretrained(..., num_labels=None)` crashes** — the
   workaround is to pass `num_labels=0` explicitly. v1 uses this.
3. **Install chain pins numpy back to 1.x** — breaks sklearn imports.
   v1 added a post-install cell that force-reinstalls `numpy>=2.2` and
   `scikit-learn scipy` against the restored numpy.

This v2 notebook carries forward those v1 fixes. The primary load path
is **HuggingFace `transformers.AutoModel` with `trust_remote_code=True`
and `num_labels=0`**. The TerraTorch path stays commented out behind a
`TRY_TERRATORCH = False` flag — flip if upstream resolves the conflict.

**If the smoke check fails, the failure point itself is the signal of
whether Prithvi inclusion is feasible for v1 paper scope.** Errors
surface with the full traceback; no silent fallback to broken state.

## What's different vs Prithvi v1

1. **Spatial split** instead of random stratified. Loads
   `/.../data/spatial_split/asset_id_to_split_v1.parquet`.
2. **3 training seeds** (314, 271, 161) varying head init + DataLoader
   shuffle only.
3. **Per-sector F1 baked in as v2 definition** (`{'n', 'macro_f1', 'acc',
   'per_class_f1_in_sector'}` per sector — matches CROMA v2 / AE v2
   schema for cross-FM aggregation).
4. **Best-val checkpoint restored before test eval** (`BEST_CKPT_BEFORE_TEST`).
5. **Aggregate output** with mean ± std + `per_seed` arrays.
6. **Old-vs-new split diagnostic** cell (transition table; expected
   ~47% changed).

## What's the same vs v1

- HF transformers loader path with `trust_remote_code=True, num_labels=0`
- Backbone `NAME = 'prithvi_eo_v2_300m_tl'`
- 6-band selection: `PRITHVI_BAND_INDICES = [2, 1, 0, 4, 5, 6]`
  (our `.npy` storage is `[B04, B03, B02, B08, B8A, B11, B12, VV, VH]`;
  Prithvi expects `[Blue=B02, Green=B03, Red=B04, NarrowNIR=B8A, SWIR1=B11, SWIR2=B12]`)
- T=1 temporal unsqueeze in the dataset → loader returns `(B, 6, 1, H, W)`
- `percentile_normalize` (2nd/98th, matches the S2 path)
- `IMAGE_SIZE = 224`
- Defensive `_probe_forward` and `_extract_features` methods (handle the
  variable output shapes Prithvi can return across transformers versions)

## Outputs (only after training; SMOKE_ONLY=True by default)

- `/.../results/fm_eval_prithvi_v2_spatial/prithvi_v2_seed{314,271,161}_results.json`
- `/.../results/fm_eval_prithvi_v2_spatial/prithvi_v2_aggregate.json`
- `/.../results/fm_eval_prithvi_v2_spatial/confusion_matrix_prithvi_v2_aggregate.png`

## Architecture note (carried from v1)

Prithvi-EO-2.0 was pretrained on Harmonized Landsat Sentinel-2 (HLS) at
30m resolution. Our evaluation uses Sentinel-2 L2A at 10m, which is a
known resolution-distribution mismatch — treated as a "transfer
scenario" in the Methods text.


In [1]:
!pip install --upgrade --force-reinstall torch==2.6.0 torchvision==0.21.0 "pillow<12" -q
!pip install -q terratorch==0.99.8 transformers==4.41.0 huggingface-hub==0.36.2 pyarrow scikit-learn scipy
!pip install --upgrade --force-reinstall --no-deps scikit-learn scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 152.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 141.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 114.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━

In [1]:
# Pre-flight: check what's actually on disk for the critical libraries.
# Pip queries the filesystem, not Python's in-memory imports, so this
# tells us the truth regardless of what's loaded.
import subprocess

critical = ['torch', 'torchvision', 'pillow', 'numpy', 'scipy',
            'scikit-learn', 'terratorch', 'transformers',
            'huggingface-hub']

for pkg in critical:
    result = subprocess.run(
        ['pip', 'show', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        # Parse the "Version: X.Y.Z" line
        for line in result.stdout.split('\n'):
            if line.startswith('Version:'):
                print(f'  {pkg:<20s} {line.split(":", 1)[1].strip()}')
                break
    else:
        print(f'  {pkg:<20s} NOT INSTALLED')

  torch                2.6.0
  torchvision          0.21.0
  pillow               11.3.0
  numpy                2.5.0
  scipy                1.18.0
  scikit-learn         1.9.0
  terratorch           0.99.8
  transformers         4.41.0
  huggingface-hub      0.36.2


In [2]:
# Conditional numpy restore. terratorch==0.99.8 pip resolution may pin
# numpy down to 1.x on some Colab images, which then breaks sklearn 1.5+.
# Only force the upgrade if we detect the downgrade. When we do upgrade,
# raise to force a clean kernel restart — pip cannot swap numpy in an
# already-loaded kernel.
import numpy as np
if np.__version__.startswith('1.'):
    print(f'numpy is on {np.__version__} (1.x). Restoring to 2.x and forcing a restart...')
    !pip install --upgrade --force-reinstall "numpy>=2.2"
    !pip install --upgrade --force-reinstall --no-deps scikit-learn scipy
    raise RuntimeError(
        'numpy was pinned to 1.x by the install above; restored to 2.x. '
        'Restart the Colab kernel (Runtime -> Restart session) and re-run '
        'from the top.'
    )
else:
    print(f'numpy {np.__version__} OK (no restore needed)')

# Also re-verify sklearn/scipy import cleanly after the install step.
import sklearn, scipy
print(f'sklearn  {sklearn.__version__}')
print(f'scipy    {scipy.__version__}')


numpy 2.5.0 OK (no restore needed)
sklearn  1.9.0
scipy    1.18.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Runtime -> Change runtime type -> GPU before training.')


Mounted at /content/drive
GPU available: True
GPU: NVIDIA L4
Memory: 23.7 GB


In [4]:
import torch
print(f"torch: {torch.__version__}")
print(f"torchvision available:", end=" ")
try:
    import torchvision
    print(f"{torchvision.__version__}")
except Exception as e:
    print(f"FAILED — {e}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
# Verify nms operator (the one that was failing)
try:
    from torchvision.ops import nms
    print("torchvision.ops.nms: OK")
except Exception as e:
    print(f"torchvision.ops.nms: FAILED — {e}")

torch: 2.6.0+cu124
torchvision available: 0.21.0+cu124
CUDA available: True
CUDA version: 12.4
torchvision.ops.nms: OK


In [5]:
# Prithvi-EO-2.0-300M-TL is public, but logging in raises rate limits
# and avoids occasional 401s on first weight download. Robust to a missing
# token — just logs a warning.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to HuggingFace.')
else:
    print('No HF_TOKEN in Colab Secrets — proceeding anonymously. '
          'Add HF_TOKEN if you hit a 401 on the weight download.')


No HF_TOKEN in Colab Secrets — proceeding anonymously. Add HF_TOKEN if you hit a 401 on the weight download.


In [6]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

if not Path(f'{EXTRACT_TO}/curation').exists():
    if Path(CODE_ZIP).exists():
        print(f'Extracting {CODE_ZIP} -> {EXTRACT_TO} ...')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(CODE_ZIP, 'r') as z:
            for member in z.namelist():
                clean = member.replace('\\', '/')
                target = os.path.join(EXTRACT_TO, clean)
                if clean.endswith('/'):
                    os.makedirs(target, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(target), exist_ok=True)
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())
        print('done.')

for candidate in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']:
    if Path(candidate).exists() and candidate not in sys.path:
        sys.path.insert(0, candidate)

# Only load_split_artifact is needed at training time. If the curation
# zip predates Phase 1, fall back to an inline definition.
try:
    from curation.utils.spatial_blocking import load_split_artifact
    print('Imported load_split_artifact from curation.utils.spatial_blocking')
except ImportError as e:
    print(f'Could not import (zip is pre-Phase-1): {e}. Using inline fallback.')
    import pandas as pd
    def load_split_artifact(input_path):
        df = pd.read_parquet(input_path, columns=['asset_id', 'split'])
        return dict(zip(df['asset_id'].astype(str), df['split'].astype(str)))


Extracting /content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip -> /content/infrabench_repo ...
done.
Could not import (zip is pre-Phase-1): No module named 'curation.utils.spatial_blocking'. Using inline fallback.


In [7]:
import os
from pathlib import Path

DATASETS_DRIVE      = f'{DRIVE_ROOT}/datasets'
DATASETS_LOCAL      = '/content/datasets'
OUTPUT_DIR          = f'{DRIVE_ROOT}/results/fm_eval_prithvi_v2_spatial'
SPLIT_ARTIFACT_PATH = f'{DRIVE_ROOT}/data/spatial_split/asset_id_to_split_v1.parquet'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'africa', 'asia', 'australia-oceania', 'central-america',
    'europe', 'north-america', 'south-america',
]
SECTORS = ['energy', 'water', 'transport', 'telecom']

CLASS_NAMES = [
    'energy.transmission.substation',
    'energy.distribution.substation',
    'energy.distribution.other',
    'energy.generation.power_plant',
    'energy.generation.solar_farm',
    'energy.generation.wind_farm',
    'water.wastewater.plant',
    'water.water_works',
    'water.storage_tank',
    'transport.airport',
    'transport.train_station',
    'transport.port_terminal',
    'telecom.data_center',
]
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'energy.transmission.substation',
    'energy.distribution.substation':         'energy.distribution.substation',
    'energy.distribution.substation_untyped': 'energy.distribution.other',
    'energy.distribution.substation_minor':   'energy.distribution.other',
    'energy.generation.power_plant':          'energy.generation.power_plant',
    'energy.generation.solar_farm':           'energy.generation.solar_farm',
    'energy.generation.wind_farm':            'energy.generation.wind_farm',
    'water.wastewater.plant':                 'water.wastewater.plant',
    'water.treatment.plant':              'water.water_works',   # legacy manifest tag
    'water.water_works':                  'water.water_works',
    'water.storage_tank':                     'water.storage_tank',
    'transport.airport':                      'transport.airport',
    'transport.train_station':                'transport.train_station',
    'transport.port_terminal':                'transport.port_terminal',
    'telecom.data_center':                    'telecom.data_center',
}

SECTOR_TO_CLASS_IDX = {
    'energy':    [0, 1, 2, 3, 4, 5],
    'water':     [6, 7, 8],
    'transport': [9, 10, 11],
    'telecom':   [12],
}

# Prithvi-specific band remap. Our .npy storage:
#   [B04, B03, B02, B08, B8A, B11, B12, VV, VH]   indices 0..8
# Prithvi expects, in order:
#   [Blue=B02, Green=B03, Red=B04, NarrowNIR=B8A, SWIR1=B11, SWIR2=B12]
# Pull our (B02=idx2, B03=idx1, B04=idx0, B8A=idx4, B11=idx5, B12=idx6):
PRITHVI_BAND_INDICES = [2, 1, 0, 4, 5, 6]

PERC_LO, PERC_HI = 2.0, 98.0
IMAGE_SIZE = 224
LP_EPOCHS  = 25
LP_BATCH   = 16
LP_LR      = 1e-3
WEIGHT_CAP = 10.0
SEEDS      = [314, 271, 161]

# Escape hatch for re-attempting TerraTorch when/if upstream fixes the
# scipy/dask/rapids conflict. Default False per v1 observation.
TRY_TERRATORCH = True


def set_seed(seed):
    import random, numpy as np
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f'Output dir:           {OUTPUT_DIR}')
print(f'Split artifact:       {SPLIT_ARTIFACT_PATH}')
print(f'Prithvi band indices: {PRITHVI_BAND_INDICES}  (B02, B03, B04, B8A, B11, B12)')
print(f'Image size:           {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'Training seeds:       {SEEDS}')
print(f'Linear probe:         {LP_EPOCHS} epochs, batch {LP_BATCH}, lr {LP_LR}')
print(f'TRY_TERRATORCH:       {TRY_TERRATORCH}  (True = TerraTorch is now the primary loader path)')


Output dir:           /content/drive/MyDrive/infra_fm/results/fm_eval_prithvi_v2_spatial
Split artifact:       /content/drive/MyDrive/infra_fm/data/spatial_split/asset_id_to_split_v1.parquet
Prithvi band indices: [2, 1, 0, 4, 5, 6]  (B02, B03, B04, B8A, B11, B12)
Image size:           224x224
Training seeds:       [314, 271, 161]
Linear probe:         25 epochs, batch 16, lr 0.001
TRY_TERRATORCH:       True  (True = TerraTorch is now the primary loader path)


In [8]:
import zipfile, shutil, time, re
from pathlib import Path

drive_path = Path(DATASETS_DRIVE)
MULTISECTOR_RE = re.compile(r'^dataset_([a-z-]+)_(energy|water|transport|telecom)_v1_1k$')


def discover_multisector_sources():
    sources, seen = [], set()
    for entry in sorted(drive_path.iterdir()):
        stem = entry.stem if entry.suffix == '.zip' else entry.name
        m = MULTISECTOR_RE.match(stem)
        if not m: continue
        region, sector = m.group(1), m.group(2)
        if region not in REGIONS: continue
        key = (region, sector)
        if key in seen: continue
        seen.add(key)
        sources.append((region, sector, entry, entry.suffix == '.zip'))
    return sources


def materialize_source(region, sector, src_path, is_zip, force=False):
    folder_name = f'dataset_{region}_{sector}_v1_1k'
    target_dir  = Path(DATASETS_LOCAL) / folder_name
    if not force and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE]   {region:<22s} {sector:<10s} already present ({n} tiles)')
        return target_dir
    if not is_zip:
        if not (src_path / 'manifest.json').exists():
            return None
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(src_path, target_dir)
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [COPY]   {region:<22s} {sector:<10s} ({n} tiles)')
        return target_dir
    t0 = time.time()
    with zipfile.ZipFile(src_path) as zf:
        names = zf.namelist()
        wrapped_prefix = f'{folder_name}/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)
        if is_wrapped:
            zf.extractall(DATASETS_LOCAL)
        else:
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'  [EXTRACT]{region:<22s} {sector:<10s} {time.time()-t0:.0f}s ({n} tiles)')
    return target_dir


discovered = discover_multisector_sources()
ready = []
for region, sector, src_path, is_zip in discovered:
    local = materialize_source(region, sector, src_path, is_zip)
    if local and (local / 'manifest.json').exists() and any((local / 'images').glob('*.npy')):
        ready.append((region, sector, local))
print(f'\nReady: {len(ready)} (region, sector) pairs')


  [EXTRACT]africa                 energy     6s (949 tiles)
  [EXTRACT]africa                 telecom    2s (1 tiles)
  [EXTRACT]africa                 transport  6s (1000 tiles)
  [EXTRACT]africa                 water      6s (892 tiles)
  [EXTRACT]asia                   energy     5s (889 tiles)
  [EXTRACT]asia                   telecom    2s (28 tiles)
  [EXTRACT]asia                   transport  6s (891 tiles)
  [EXTRACT]asia                   water      5s (958 tiles)
  [EXTRACT]australia-oceania      energy     6s (1002 tiles)
  [EXTRACT]australia-oceania      telecom    2s (12 tiles)
  [EXTRACT]australia-oceania      transport  7s (1000 tiles)
  [EXTRACT]australia-oceania      water      6s (1000 tiles)
  [EXTRACT]central-america        energy     6s (999 tiles)
  [EXTRACT]central-america        telecom    2s (1 tiles)
  [EXTRACT]central-america        transport  7s (566 tiles)
  [EXTRACT]central-america        water      6s (999 tiles)
  [EXTRACT]europe                 energy  

In [9]:
import json
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from collections import Counter
import random


@dataclass
class _Record:
    path: 'Path'
    asset_id: str
    asset_type: str


def percentile_normalize(arr, lo=PERC_LO, hi=PERC_HI):
    img = arr.astype(np.float32)
    low = np.percentile(img, lo); high = np.percentile(img, hi)
    if high <= low:
        return np.clip(img / 255.0, 0.0, 1.0)
    return np.clip((img - low) / (high - low), 0.0, 1.0)


class PrithviDataset(Dataset):
    """Self-contained loader for a single `dataset_<region>_<sector>_v1_1k/`
    folder. Selects the 6 Prithvi bands per `PRITHVI_BAND_INDICES`, applies
    percentile_normalize, and unsqueezes a T=1 temporal dim so the default
    collate stacks to `(B, 6, 1, H, W)` (Prithvi's 3D patch embedding
    expects 5D input)."""
    def __init__(self, dataset_root,
                 band_indices=PRITHVI_BAND_INDICES,
                 allowed_asset_types=tuple(ASSET_TYPE_MAP.keys())):
        self.dataset_root = Path(dataset_root)
        self.band_indices = list(band_indices)
        self.allowed = set(allowed_asset_types)
        self.max_required_band = max(self.band_indices)

        manifest_path = self.dataset_root / 'manifest.json'
        if not manifest_path.exists():
            raise FileNotFoundError(f'missing manifest.json: {manifest_path}')
        with manifest_path.open() as f:
            manifest = json.load(f)
        records_in = manifest.get('records', [])
        images_dir = self.dataset_root / 'images'

        records, dropped = [], Counter()
        for r in records_in:
            at = r.get('asset_type')
            if not at or at not in self.allowed:
                dropped['filtered_type' if at else 'no_label'] += 1; continue
            img_file = r.get('image_file')
            if not img_file:
                dropped['no_image_file'] += 1; continue
            p = images_dir / img_file
            if not p.exists():
                dropped['missing_npy'] += 1; continue
            try:
                arr = np.load(p, mmap_mode='r')
                if arr.shape[0] < self.max_required_band + 1:
                    dropped['too_few_bands'] += 1; continue
            except Exception:
                dropped['load_error'] += 1; continue
            records.append(_Record(path=p, asset_id=str(r.get('asset_id', p.stem)),
                                   asset_type=at))
        if dropped:
            print(f'  PrithviDataset({self.dataset_root.name}): dropped '
                  f'{sum(dropped.values())} records ({dict(dropped)})')
        if not records:
            raise RuntimeError(f'no usable records in {self.dataset_root}')
        self.records = records

    def __len__(self): return len(self.records)

    def _load_image(self, path):
        arr = np.load(path)
        arr = arr[self.band_indices, :, :]              # (6, H, W) in Prithvi band order
        arr = percentile_normalize(arr)                  # -> [0, 1]
        return arr.astype(np.float32)

    def __getitem__(self, idx):
        r = self.records[idx]
        img = self._load_image(r.path)                   # (6, H, W)
        t = torch.from_numpy(img).unsqueeze(1)           # (6, 1, H, W)  — T=1 dim
        return {'image': t, 'asset_id': r.asset_id, 'asset_type': r.asset_type}


class MultiSectorLabelWrapper(Dataset):
    def __init__(self, base, region, sector, input_size=IMAGE_SIZE):
        self.base = base; self.region = region; self.sector = sector
        self.input_size = input_size
        self.valid_indices, self.labels, self.asset_ids = [], [], []
        for i, r in enumerate(base.records):
            mapped = ASSET_TYPE_MAP.get(r.asset_type)
            if mapped is None: continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])
            self.asset_ids.append(r.asset_id)

    def __len__(self): return len(self.valid_indices)

    def __getitem__(self, idx):
        sample = self.base[self.valid_indices[idx]]
        img = sample['image']                            # (6, 1, H, W)
        C, T, H, W = img.shape
        img = img.reshape(C * T, 1, H, W)
        img = F.interpolate(img, size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False)
        img = img.reshape(C, T, self.input_size, self.input_size)
        return {'image': img, 'label': self.labels[idx],
                'asset_id': sample['asset_id'],
                'region': self.region, 'sector': self.sector}


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base; self.indices = indices
        self.region = base.region; self.sector = base.sector
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


source_datasets = {}
for region, sector, local in ready:
    base = PrithviDataset(local)
    ds = MultiSectorLabelWrapper(base, region=region, sector=sector)
    if len(ds): source_datasets[(region, sector)] = ds
print(f'Built {len(source_datasets)} cell datasets')


Built 28 cell datasets


In [10]:
# Load the spatial split artifact and slice each cell into train/val/test.
asset_to_split = load_split_artifact(SPLIT_ARTIFACT_PATH)
print(f'Loaded split artifact: {len(asset_to_split):,} asset_id -> split entries')
print(f'  splits distribution: {Counter(asset_to_split.values())}')

splits = {}
n_unmapped = 0
for key, ds in source_datasets.items():
    region, sector = key
    tr_idx, va_idx, te_idx = [], [], []
    for i in range(len(ds)):
        asset_id = ds.asset_ids[i]
        sp = asset_to_split.get(asset_id)
        if sp == 'train':   tr_idx.append(i)
        elif sp == 'val':   va_idx.append(i)
        elif sp == 'test':  te_idx.append(i)
        else:                n_unmapped += 1
    splits[key] = {
        'train': SubsetView(ds, tr_idx),
        'val':   SubsetView(ds, va_idx),
        'test':  SubsetView(ds, te_idx),
    }

if n_unmapped > 0:
    print(f'NOTE: {n_unmapped} tiles have no split assignment (excluded from train/val/test).')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')


Loaded split artifact: 18,750 asset_id -> split entries
  splits distribution: Counter({'train': 13087, 'val': 2851, 'test': 2812})

Global: train=13087  val=2856  test=2813


In [11]:
# Diagnostic: regenerate the v1 random stratified split inside the
# notebook for an exact-match comparison vs the new spatial split.
print('=' * 76)
print('Diagnostic: train/val/test transition table (old random -> new spatial)')
print('=' * 76)


def old_stratified_split(dataset_labels, train_frac=0.7, val_frac=0.15, seed=42):
    by_class = {}
    for i, label in enumerate(dataset_labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    tr, va, te = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs); n_tr = int(n * train_frac); n_va = int(n * val_frac)
        tr.extend(idxs[:n_tr])
        va.extend(idxs[n_tr:n_tr + n_va])
        te.extend(idxs[n_tr + n_va:])
    return tr, va, te


old_split = {}
for key in sorted(source_datasets):
    ds = source_datasets[key]
    tr, va, te = old_stratified_split(ds.labels)
    for i in tr: old_split[ds.asset_ids[i]] = 'train'
    for i in va: old_split[ds.asset_ids[i]] = 'val'
    for i in te: old_split[ds.asset_ids[i]] = 'test'

common_ids = set(old_split) & set(asset_to_split)
print(f'Comparing on {len(common_ids):,} tiles in both old and new splits')
counts = Counter()
for aid in common_ids:
    counts[(old_split[aid], asset_to_split[aid])] += 1

print(f'\nold \\ new      train       val       test')
print('-' * 50)
for old_sp in ('train', 'val', 'test'):
    row = [counts.get((old_sp, new_sp), 0) for new_sp in ('train', 'val', 'test')]
    print(f'{old_sp:<8s}  {row[0]:>10d} {row[1]:>9d} {row[2]:>10d}')

same    = sum(counts.get((sp, sp), 0) for sp in ('train', 'val', 'test'))
changed = len(common_ids) - same
print(f'\nUnchanged: {same:,} ({100.0*same/len(common_ids):.1f}%)')
print(f'Changed:   {changed:,} ({100.0*changed/len(common_ids):.1f}%)')
print('\n(Expected ~47% changed — same spatial split as CROMA v2 / AE v2 / SatlasS2 v2 / SatlasS1 v2.)')


Diagnostic: train/val/test transition table (old random -> new spatial)
Comparing on 18,750 tiles in both old and new splits

old \ new      train       val       test
--------------------------------------------------
train           9120      1983       1978
val             1946       410        415
test            2021       458        419

Unchanged: 9,949 (53.1%)
Changed:   8,801 (46.9%)

(Expected ~47% changed — same spatial split as CROMA v2 / AE v2 / SatlasS2 v2 / SatlasS1 v2.)


In [12]:
import torch.nn as nn


class PrithviBackbone(nn.Module):
    """Prithvi-EO-2.0-300M-TL frozen backbone, mean-pooled patch features.

    LOADER STRATEGY (v2 — TerraTorch primary)
    -----------------------------------------
    Per the IBM/NASA HuggingFace model card, TerraTorch is the canonical
    load path for Prithvi-EO-2.0. We try three TerraTorch entry points in
    order, then fall back to HF transformers if all three fail.

      1. `terratorch.registry.BACKBONE_REGISTRY.build('prithvi_eo_v2_300_tl', pretrained=True)`
         — the lowest-level primitive. Returns just the encoder.
      2. `terratorch.models.backbones.prithvi_select.prithvi_eo_v2_300_tl(pretrained=True)`
         — the timm-style factory function. Some terratorch versions expose
         only this path under `prithvi_select` (or `prithvi_vit`); the
         attempt below tries multiple submodule names.
      3. `terratorch.tasks.SemanticSegmentationTask(...).model.encoder`
         — build the full task wrapper (as the blumenstiel reference notebook
         does) then extract the encoder. Heavier but uses the high-level
         path the TerraTorch team specifically maintains.
      4. `transformers.AutoModel.from_pretrained(repo, trust_remote_code=True, num_labels=0)`
         — the v1 fallback. Currently known broken in this environment;
         included so the all-paths-failed diagnostic shows the full picture.

    If all four fail, we raise a single RuntimeError that lists each
    attempt's exception. The failure-pattern itself is the operative
    signal for whether Prithvi inclusion is feasible.

    FEATURE EXTRACTION (carried verbatim from v1)
    ---------------------------------------------
    Prithvi's encoder returns patch-token sequences `(B, N, D)`. We
    mean-pool over the token dim to get `(B, D)`. The forward is
    defensive against tuple/list/`ModelOutput`/3D/4D/5D return shapes.
    """
    NAME = 'prithvi_eo_v2_300m_tl'
    HF_REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
    EXPECTED_FEATURE_DIM = 1024

    def __init__(self, freeze=True):
        super().__init__()
        self._loader_used = None
        self._load_errors = []     # list of (attempt_name, exception_repr)
        self.backbone = self._load_backbone()
        self.feature_dim = self._infer_feature_dim()
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    # ------------------------------------------------------------------ #
    # Loader attempts
    # ------------------------------------------------------------------ #

    def _try_backbone_registry(self):
        from terratorch.registry import BACKBONE_REGISTRY
        bb = BACKBONE_REGISTRY.build('prithvi_eo_v2_300_tl', pretrained=True)
        return bb

    def _try_timm_style_factory(self):
        # Several terratorch versions stash the timm-style factory under
        # different submodules. Try the common ones.
        candidate_modules = [
            'terratorch.models.backbones.prithvi_select',
            'terratorch.models.backbones.prithvi_vit',
            'terratorch.models.backbones',
        ]
        last_err = None
        for mod_path in candidate_modules:
            try:
                mod = __import__(mod_path, fromlist=['prithvi_eo_v2_300_tl'])
                factory = getattr(mod, 'prithvi_eo_v2_300_tl', None)
                if factory is None:
                    last_err = AttributeError(
                        f'{mod_path} has no prithvi_eo_v2_300_tl attribute'
                    )
                    continue
                return factory(pretrained=True)
            except Exception as e:
                last_err = e
                continue
        if last_err is not None:
            raise last_err
        raise RuntimeError('No candidate module exposed prithvi_eo_v2_300_tl')

    def _try_segmentation_task_extract(self):
        import terratorch
        # Build the full task (matching the blumenstiel reference's pattern)
        # then extract the encoder. UNetDecoder + 2-class head is arbitrary;
        # we discard everything except the encoder.
        task = terratorch.tasks.SemanticSegmentationTask(
            model_factory='EncoderDecoderFactory',
            model_args={
                'backbone': 'prithvi_eo_v2_300_tl',
                'backbone_pretrained': True,
                'decoder': 'UNetDecoder',
                'decoder_channels': [256, 128, 64, 32],
                'num_classes': 2,
            },
            loss='ce',
            ignore_index=-1,
        )
        # Probe likely attribute paths for the encoder.
        candidates = [
            ('task.model.encoder',  lambda: task.model.encoder),
            ('task.encoder',        lambda: task.encoder),
            ('task.model.backbone', lambda: task.model.backbone),
            ('task.model',          lambda: task.model),
        ]
        for label, getter in candidates:
            try:
                obj = getter()
                if obj is None:
                    continue
                # Sanity-check: does it look like a Prithvi encoder?
                # Just confirm it has parameters (a real nn.Module).
                if sum(p.numel() for p in obj.parameters()) > 1e5:
                    print(f'    extracted encoder via {label}')
                    return obj
            except AttributeError:
                continue
        # If nothing matched, list what's on task.model so the diagnostic
        # is informative.
        attrs = [a for a in dir(task.model) if not a.startswith('_')]
        raise RuntimeError(
            f'SemanticSegmentationTask built OK but no encoder attribute '
            f'found via {[c[0] for c in candidates]}. task.model attributes: '
            f'{attrs[:30]}{"..." if len(attrs) > 30 else ""}'
        )

    def _try_transformers_fallback(self):
        from transformers import AutoModel
        return AutoModel.from_pretrained(
            self.HF_REPO,
            trust_remote_code=True,
            num_labels=0,
        )

    def _load_backbone(self):
        attempts = [
            ('terratorch_registry',         self._try_backbone_registry),
            ('terratorch_timm_factory',     self._try_timm_style_factory),
            ('terratorch_task_extract',     self._try_segmentation_task_extract),
            ('transformers_automodel',      self._try_transformers_fallback),
        ]
        for name, fn in attempts:
            print(f'  attempt {name} ...', flush=True)
            try:
                bb = fn()
                print(f'    SUCCESS via {name}')
                self._loader_used = name
                return bb
            except Exception as e:
                msg = f'{e.__class__.__name__}: {e}'
                print(f'    FAILED: {msg}')
                self._load_errors.append((name, msg))

        # All four failed — surface the full picture.
        lines = ['', '=' * 76,
                 'Prithvi backbone load FAILED — all four loader paths exhausted',
                 '=' * 76]
        for name, msg in self._load_errors:
            lines.append(f'  [{name}]')
            for ln in msg.splitlines():
                lines.append(f'    {ln}')
        lines.extend([
            '',
            'Diagnostic next steps:',
            '  - If terratorch_registry / timm_factory / task_extract all hit the same',
            '    scipy/dask error: TerraTorch 0.99.8 is broken in this Colab image.',
            '    Either pin to a different terratorch version or defer Prithvi to v2 paper.',
            '  - If transformers_automodel hit num_labels: pin transformers==4.41.0 OR upgrade beyond.',
            '  - If HF Hub 401: huggingface-cli login or set HF_TOKEN.',
            '  - The error pattern above is the signal we need to decide whether',
            '    Prithvi inclusion is feasible for v1 paper scope.',
            '=' * 76,
        ])
        raise RuntimeError('\n'.join(lines))

    # ------------------------------------------------------------------ #
    # Forward signature probing + feature extraction (unchanged from v1)
    # ------------------------------------------------------------------ #

    def _probe_forward(self, dummy):
        attempts = [
            lambda: self.backbone(dummy),
            lambda: self.backbone(dummy, temporal_coords=None, location_coords=None),
            lambda: self.backbone(pixel_values=dummy),
            lambda: self.backbone(pixel_values=dummy, temporal_coords=None, location_coords=None),
        ]
        last_err = None
        for attempt in attempts:
            try:
                return attempt()
            except (TypeError, RuntimeError) as e:
                last_err = e
        raise RuntimeError(f'No Prithvi forward signature worked. Last error: {last_err}')

    @staticmethod
    def _extract_features(out):
        if isinstance(out, (tuple, list)):
            out = out[-1]
        if hasattr(out, 'last_hidden_state'):
            out = out.last_hidden_state
        if out.dim() == 3:    return out.mean(dim=1)
        if out.dim() == 4:    return out.mean(dim=[2, 3])
        if out.dim() == 5:    return out.mean(dim=[2, 3, 4])
        raise RuntimeError(f'Unexpected Prithvi output shape: {tuple(out.shape)}')

    def _infer_feature_dim(self):
        self.backbone.eval()
        device = next(self.backbone.parameters()).device
        dummy = torch.zeros(1, 6, 1, IMAGE_SIZE, IMAGE_SIZE, device=device)
        with torch.no_grad():
            out = self._probe_forward(dummy)
            feat = self._extract_features(out)
        d = feat.shape[-1]
        if d != self.EXPECTED_FEATURE_DIM:
            print(f'  WARNING: feature_dim={d} (expected {self.EXPECTED_FEATURE_DIM} '
                  f'for 300M variant). Using observed dim.')
        else:
            print(f'  feature_dim = {d} (matches expected for Prithvi-EO-2.0 300M)')
        return d

    def forward(self, x):
        if x.dim() == 4:
            x = x.unsqueeze(2)
        out = self._probe_forward(x)
        return self._extract_features(out)


class InfraBenchClassifier(nn.Module):
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(nn.Dropout(dropout),
                                  nn.Linear(backbone.feature_dim, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


def BACKBONE_FACTORY(freeze=True):
    return PrithviBackbone(freeze=freeze)


print('PrithviBackbone + InfraBenchClassifier defined.')


PrithviBackbone + InfraBenchClassifier defined.


In [13]:
from torch.optim import AdamW
from collections import defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set, max_weight=WEIGHT_CAP):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        if counts.get(c, 0) > 0:
            w = total / (len(CLASS_NAMES) * counts[c])
            weights[c] = min(w, max_weight)
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),   # (B, 6, 1, H, W)
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
        'sector': [b['sector'] for b in batch],
    }


def _per_sector_v2(y_true, y_pred, per_class_f1, cm):
    """v2 per-sector — matches CROMA v2 schema exactly.
    Mean of per-class F1s for the classes in that sector, computed on the
    FULL test set."""
    cm = np.array(cm)
    out = {}
    for sector, class_idxs in SECTOR_TO_CLASS_IDX.items():
        sector_f1s = [float(per_class_f1[i]) for i in class_idxs]
        macro_f1 = float(np.mean(sector_f1s)) if sector_f1s else 0.0
        n_sector = int(sum(cm[i, :].sum() for i in class_idxs))
        correct  = int(sum(cm[i, i]      for i in class_idxs))
        acc = correct / n_sector if n_sector > 0 else float('nan')
        out[sector] = {
            'n': n_sector,
            'macro_f1': macro_f1,
            'acc': acc,
            'per_class_f1_in_sector': {
                CLASS_NAMES[i]: sector_f1s[idx]
                for idx, i in enumerate(class_idxs)
            },
        }
    return out


@torch.no_grad()
def evaluate(model, loader, return_breakdowns=False):
    model.eval()
    all_preds, all_labels, all_regions, all_sectors = [], [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])
        all_sectors.extend(batch['sector'])

    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                            labels=list(range(len(CLASS_NAMES))),
                            zero_division=0.0).tolist()
    cm = confusion_matrix(all_labels, all_preds,
                          labels=list(range(len(CLASS_NAMES)))).tolist()
    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': per_class_f1,
        'confusion': cm,
    }
    if return_breakdowns:
        def grouped_region_f1():
            groups = defaultdict(lambda: {'preds': [], 'labels': []})
            for p, l, g in zip(all_preds, all_labels, all_regions):
                groups[g]['preds'].append(p)
                groups[g]['labels'].append(l)
            return {
                g: {
                    'n': len(d['labels']),
                    'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro',
                                               zero_division=0.0)),
                    'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
                }
                for g, d in groups.items()
            }
        result['per_region']                 = grouped_region_f1()
        result['per_sector']                 = _per_sector_v2(all_labels, all_preds, per_class_f1, cm)
        result['per_sector_corrected']       = result['per_sector']
        result['per_sector_correction_note'] = (
            'per_sector = macro-average of per-class F1s for classes in that '
            'sector, computed on the FULL test set (v2 definition).'
        )
    return result


def train_one_seed(seed, *, train_set, val_set, test_set):
    set_seed(seed)
    print(f'\n--- seed {seed} ---')
    backbone = BACKBONE_FACTORY(freeze=True)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_set, batch_size=LP_BATCH, shuffle=True,
                              num_workers=2, collate_fn=collate, pin_memory=True,
                              generator=g)
    val_loader   = DataLoader(val_set,   batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=LP_BATCH, shuffle=False,
                              num_workers=2, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                      lr=LP_LR, weight_decay=1e-4)

    run_name = f'prithvi_v2_seed{seed}_linear_probe'
    ckpt_dir = Path(OUTPUT_DIR) / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt  = ckpt_dir / 'checkpoint_best.pt'
    final_ckpt = ckpt_dir / 'checkpoint_final.pt'

    history, best_val_f1, best_epoch = [], -1.0, -1
    for epoch in range(LP_EPOCHS):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        })
        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch + 1
            marker = ' *'
            torch.save({'epoch': epoch + 1,
                        'model_state_dict': model.state_dict(),
                        'val_macro_f1': val['macro_f1'],
                        'history': history}, best_ckpt)
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    torch.save({'epoch': LP_EPOCHS,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'history': history, 'best_val_f1': best_val_f1,
                'best_epoch': best_epoch}, final_ckpt)

    # ============== BEST_CKPT_BEFORE_TEST =================================
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'  [BEST_CKPT_BEFORE_TEST] restored epoch {ckpt["epoch"]} '
              f'(val_f1={ckpt["val_macro_f1"]:.4f}) before test')
        tested_with = f'best-val checkpoint from epoch {best_epoch}'
    else:
        tested_with = 'final-epoch state'

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_breakdowns=True)
    return {
        'run_name':     run_name,
        'backbone':     backbone.NAME,
        'condition':    'linear_probe',
        'num_epochs':   LP_EPOCHS,
        'seed':         seed,
        'best_val_f1':  best_val_f1,
        'best_epoch':   best_epoch,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1':  float(np.std(tail)),
        'history':      history,
        'test':         test,
        'tested_with':  tested_with,
    }


print('Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).')


Device: cuda
Training infrastructure ready (v2 per-sector F1, BEST_CKPT_BEFORE_TEST, multi-seed).


In [15]:
# Default SMOKE_ONLY=True. If smoke fails at any specific point, that
# failure IS the diagnostic. Flip to False once smoke is happy.
SMOKE_ONLY = False

print('Loading Prithvi backbone (the historically fragile step)...')
set_seed(SEEDS[0])
try:
    sb = PrithviBackbone(freeze=True)
except Exception as e:
    print()
    print('=' * 76)
    print('SMOKE CHECK FAILED at backbone load.')
    print('=' * 76)
    print('See the traceback above. Per the v2 plan, this failure point is')
    print("the signal we needed — it tells us whether Prithvi inclusion is")
    print('feasible for v1 paper scope or genuinely needs to be deferred.')
    raise

sm = InfraBenchClassifier(sb, num_classes=len(CLASS_NAMES)).to(DEVICE)
print(f'  loader_used = {sb._loader_used}')
print(f'  feature_dim = {sb.feature_dim}')
print(f'  trainable params = {sum(p.numel() for p in sm.parameters() if p.requires_grad):,}')

smoke_loader = DataLoader(train_global, batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate)
batch = next(iter(smoke_loader))
img = batch['image']
print(f'\n  Batch shape: {tuple(img.shape)}  (expect [4, 6, 1, {IMAGE_SIZE}, {IMAGE_SIZE}])')
print(f'  Batch range: [{img.min().item():.4f}, {img.max().item():.4f}]  (expect ~[0, 1])')

print('\nForward pass...')
sm.eval()
with torch.no_grad():
    feats = sb(img.to(DEVICE))
    logits = sm(img.to(DEVICE))
print(f'  Backbone features: {tuple(feats.shape)}  (expect [4, {sb.feature_dim}])')
print(f'  Classifier logits: {tuple(logits.shape)}  (expect [4, {len(CLASS_NAMES)}])')
print(f'  Logits finite: {torch.isfinite(logits).all().item()}')
assert torch.isfinite(logits).all().item(), 'NaN/Inf in logits — aborting smoke'

print('\nOne training step...')
sm.train()
weights = compute_class_weights(train_global).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = AdamW([p for p in sm.parameters() if p.requires_grad],
                  lr=LP_LR, weight_decay=1e-4)
optimizer.zero_grad()
loss = criterion(sm(img.to(DEVICE)), batch['label'].to(DEVICE))
loss.backward()
head_grads = [p.grad for p in sm.head.parameters() if p.grad is not None]
assert head_grads and any(g.abs().sum().item() > 0 for g in head_grads), \
    'head received zero gradients'
optimizer.step()
print(f'  Train step loss: {loss.item():.4f}  (finite={torch.isfinite(loss).item()})')

print(f'\nSmoke check PASSED. SMOKE_ONLY = {SMOKE_ONLY} — multi-seed cell below will '
      f'{"NOT train" if SMOKE_ONLY else "run all 3 seeds"}.')


Loading Prithvi backbone (the historically fragile step)...
  attempt terratorch_registry ...
    SUCCESS via terratorch_registry
  feature_dim = 1024 (matches expected for Prithvi-EO-2.0 300M)
  loader_used = terratorch_registry
  feature_dim = 1024
  trainable params = 13,325

  Batch shape: (4, 6, 1, 224, 224)  (expect [4, 6, 1, 224, 224])
  Batch range: [0.0000, 1.0000]  (expect ~[0, 1])

Forward pass...
  Backbone features: (4, 1024)  (expect [4, 1024])
  Classifier logits: (4, 13)  (expect [4, 13])
  Logits finite: True

One training step...
  Train step loss: 1.7237  (finite=True)

Smoke check PASSED. SMOKE_ONLY = False — multi-seed cell below will run all 3 seeds.


In [16]:
import json as _json
import numpy as np

if SMOKE_ONLY:
    print('SMOKE_ONLY=True — skipping all training. Set False and re-run.')
else:
    per_seed_results = {}
    for seed in SEEDS:
        result = train_one_seed(seed,
                                train_set=train_global,
                                val_set=val_global,
                                test_set=test_global)
        per_seed_results[seed] = result
        out_path = Path(OUTPUT_DIR) / f'prithvi_v2_seed{seed}_results.json'
        with open(out_path, 'w') as f:
            _json.dump({'linear_probe': result}, f, indent=2)
        print(f'\n  saved {out_path}')

    def _agg(values):
        arr = np.array(values, dtype=np.float64)
        return {'mean': float(arr.mean()), 'std': float(arr.std(ddof=0)),
                'per_seed': [float(v) for v in arr]}

    agg = {}
    agg['test_macro_f1'] = _agg([per_seed_results[s]['test']['macro_f1'] for s in SEEDS])
    agg['test_accuracy'] = _agg([per_seed_results[s]['test']['acc'] for s in SEEDS])

    per_class_arr = np.array([per_seed_results[s]['test']['per_class_f1'] for s in SEEDS])
    agg['per_class_f1'] = [
        {'class': CLASS_NAMES[i], 'idx': i,
         'mean_f1': float(per_class_arr[:, i].mean()),
         'std_f1':  float(per_class_arr[:, i].std(ddof=0)),
         'per_seed': [float(v) for v in per_class_arr[:, i]]}
        for i in range(len(CLASS_NAMES))
    ]

    agg['per_sector_f1'] = {}
    for sector in SECTOR_TO_CLASS_IDX:
        f1s = [per_seed_results[s]['test']['per_sector'][sector]['macro_f1'] for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_sector'][sector]['n']
        agg['per_sector_f1'][sector] = {
            'n': n_seed,
            'mean_macro_f1': float(np.mean(f1s)),
            'std_macro_f1':  float(np.std(f1s, ddof=0)),
            'per_seed': [float(v) for v in f1s],
        }

    region_keys = sorted({r for s in SEEDS for r in per_seed_results[s]['test']['per_region']})
    agg['per_region_f1'] = {}
    for region in region_keys:
        f1s = [per_seed_results[s]['test']['per_region'].get(region, {}).get('macro_f1', float('nan'))
               for s in SEEDS]
        n_seed = per_seed_results[SEEDS[0]]['test']['per_region'].get(region, {}).get('n', 0)
        agg['per_region_f1'][region] = {
            'n': n_seed,
            'mean_macro_f1': float(np.nanmean(f1s)),
            'std_macro_f1':  float(np.nanstd(f1s, ddof=0)),
            'per_seed': [float(v) for v in f1s],
        }

    agg['seeds']                       = SEEDS
    agg['split_artifact']              = SPLIT_ARTIFACT_PATH
    agg['per_sector_correction_note']  = per_seed_results[SEEDS[0]]['test']['per_sector_correction_note']

    agg_path = Path(OUTPUT_DIR) / 'prithvi_v2_aggregate.json'
    with open(agg_path, 'w') as f:
        _json.dump(agg, f, indent=2)
    print(f'\nAggregate saved: {agg_path}')

    # Confusion matrix PNG
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    cm_sum = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
    for s in SEEDS:
        cm_sum += np.array(per_seed_results[s]['test']['confusion'], dtype=np.int64)
    cm_norm = cm_sum.astype(np.float64)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_norm, row_sums, out=np.zeros_like(cm_norm), where=row_sums > 0)

    short_names = [n.split('.', 1)[1] if '.' in n else n for n in CLASS_NAMES]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm_norm, cmap='Oranges', vmin=0, vmax=1)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(short_names, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    mf1 = agg['test_macro_f1']
    ax.set_title(f'Prithvi v2 spatial — aggregate confusion (summed over '
                 f'{len(SEEDS)} seeds, row-normalized)\n'
                 f'macro F1 = {mf1["mean"]:.3f} +/- {mf1["std"]:.3f}, seeds = {SEEDS}')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            v = cm_norm[i, j]
            if v > 0.01:
                color = 'white' if v > 0.5 else 'black'
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', color=color, fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    cm_path = Path(OUTPUT_DIR) / 'confusion_matrix_prithvi_v2_aggregate.png'
    plt.savefig(cm_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Confusion matrix saved: {cm_path}')

    print('\n' + '=' * 76)
    print(f'Prithvi v2 spatial — aggregate ({len(SEEDS)} seeds)')
    print('=' * 76)
    print(f'Test macro F1: {agg["test_macro_f1"]["mean"]:.4f} +/- '
          f'{agg["test_macro_f1"]["std"]:.4f}')
    print(f'Test accuracy: {agg["test_accuracy"]["mean"]:.4f} +/- '
          f'{agg["test_accuracy"]["std"]:.4f}')
    print('\nPer-class F1 (mean +/- std):')
    for entry in agg['per_class_f1']:
        print(f'  [{entry["idx"]:>2d}] {entry["class"]:<34s} '
              f'{entry["mean_f1"]:.4f} +/- {entry["std_f1"]:.4f}')
    print('\nPer-sector F1 (v2):')
    for sector, stats in agg['per_sector_f1'].items():
        print(f'  {sector:<10s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')
    print('\nPer-region F1:')
    for region, stats in agg['per_region_f1'].items():
        print(f'  {region:<22s} n={stats["n"]:>5d}  '
              f'{stats["mean_macro_f1"]:.4f} +/- {stats["std_macro_f1"]:.4f}')



--- seed 314 ---
  attempt terratorch_registry ...
    SUCCESS via terratorch_registry
  feature_dim = 1024 (matches expected for Prithvi-EO-2.0 300M)
  ep   1  loss=2.2334  val_acc=0.2265  val_f1=0.1282 *
  ep   2  loss=2.0898  val_acc=0.2174  val_f1=0.1522 *
  ep   3  loss=2.0402  val_acc=0.2766  val_f1=0.1816 *
  ep   4  loss=1.9939  val_acc=0.3179  val_f1=0.1789
  ep   5  loss=1.9667  val_acc=0.2868  val_f1=0.1974 *
  ep   6  loss=1.9572  val_acc=0.3029  val_f1=0.1850
  ep   7  loss=1.9379  val_acc=0.2997  val_f1=0.2032 *
  ep   8  loss=1.9229  val_acc=0.3596  val_f1=0.2228 *
  ep   9  loss=1.9146  val_acc=0.3235  val_f1=0.2103
  ep  10  loss=1.9023  val_acc=0.3134  val_f1=0.2058
  ep  11  loss=1.8966  val_acc=0.2738  val_f1=0.1850
  ep  12  loss=1.8766  val_acc=0.3071  val_f1=0.2023
  ep  13  loss=1.8721  val_acc=0.2808  val_f1=0.1960
  ep  14  loss=1.8615  val_acc=0.2665  val_f1=0.1973
  ep  15  loss=1.8551  val_acc=0.2637  val_f1=0.1923
  ep  16  loss=1.8505  val_acc=0.3449  va